In [ ]:
展示所有report.txt的内容和  对应case和检查名，以便于检查内容

In [1]:
from pathlib import Path
import pandas as pd

# =========================
# 1. 设置根目录
# =========================
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_4\raw")


# =========================
# 2. 查找所有 report.txt
# =========================
report_files = list(root_dir.rglob("report.txt"))

records = []

for report_path in sorted(report_files):
    try:
        content = report_path.read_text(encoding="utf-8").strip()
    except UnicodeDecodeError:
        try:
            content = report_path.read_text(encoding="gbk").strip()
        except Exception as e:
            content = f"[读取失败] {e}"
    except Exception as e:
        content = f"[读取失败] {e}"

    # 从路径中提取 case 名和检查名
    parts = report_path.parts

    case_name = ""
    exam_name = ""

    for part in parts:
        if part.startswith("case_"):
            case_name = part

    # report.txt 的父文件夹一般就是检查文件夹，例如 202506061036
    exam_name = report_path.parent.name

    records.append({
        "case": case_name,
        "检查文件夹": exam_name,
        "report路径": str(report_path),
        "report内容": content
    })


# =========================
# 3. 转成表格展示
# =========================
df_reports = pd.DataFrame(records)

print(f"共找到 {len(df_reports)} 个 report.txt 文件。")

df_reports

In [ ]:
##修正_x000D_乱码问题

from pathlib import Path

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

report_files = list(root_dir.rglob("report.txt"))

fix_count = 0

for report_path in report_files:
    try:
        text = report_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = report_path.read_text(encoding="gbk")

    new_text = (
        text.replace("_x000D_", "\n")
            .replace("_x000d_", "\n")
            .replace("\r\n", "\n")
            .replace("\r", "\n")
            .strip()
    )

    if new_text != text:
        report_path.write_text(new_text, encoding="utf-8")
        fix_count += 1
        print(f"已修复: {report_path}")

print(f"\n修复完成，共修复 {fix_count} 个 report.txt")

In [ ]:
##删除report中的空行

from pathlib import Path

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

def remove_blank_lines(text):
    """
    删除 report.txt 中的所有空行。
    同时兼容 _x000D_ 和不同系统换行符。
    """
    text = text.replace("_x000D_", "\n")
    text = text.replace("_x000d_", "\n")
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # 只保留非空行
    lines = [line.strip() for line in text.split("\n") if line.strip() != ""]

    return "\n".join(lines)

report_files = list(root_dir.rglob("report.txt"))

fix_count = 0

for report_path in report_files:
    try:
        old_text = report_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        old_text = report_path.read_text(encoding="gbk")

    new_text = remove_blank_lines(old_text)

    if new_text != old_text:
        report_path.write_text(new_text, encoding="utf-8")
        fix_count += 1
        print(f"已清理空行: {report_path}")

print(f"\n处理完成，共清理 {fix_count} 个 report.txt")